# Dataset preparation

In [ ]:
! pip install transformers datasets seqeval evaluate accelerate torch

In [ ]:
from datasets import load_dataset
import unicodedata
import copy
import os
import torch
from torch.utils.data.dataset import Dataset
import json
import pandas as pd

In [ ]:
DIACRITIC_MAP = str.maketrans({
    'ă': 'a', 'Ă': 'A',
    'â': 'a', 'Â': 'A',
    'î': 'i', 'Î': 'I',
    'ș': 's', 'Ș': 'S',
    'ţ': 't', 'Ţ': 'T',
    'ț': 't', 'Ț': 'T',
    'ş': 'S', 'Ş': 'S',
})

def strip_romanian_diacritics_from_tokens(tokens_list: list) -> list:
    return [text.translate(DIACRITIC_MAP) for text in tokens_list]

def strip_romanian_diacritics_from_dataset_ner(dataset: list):
    dataset_without_diac = []

    for doc in dataset:
      stripped_doc = copy.deepcopy(doc)
      stripped_doc['tokens'] = strip_romanian_diacritics_from_tokens(stripped_doc['tokens'])
      dataset_without_diac.append(stripped_doc)

    return dataset_without_diac

def strip_romanian_diacritics_from_dataset_sts(dataset: list):
    dataset_without_diac = []

    for doc in dataset:
      doc['Sent1'] = strip_romanian_diacritics_from_tokens([doc['Sent1']])[0]
      doc['Sent2'] = strip_romanian_diacritics_from_tokens([doc['Sent2']])[0]
      dataset_without_diac.append(doc)

    return dataset_without_diac

In [ ]:
data_files = {'train': 'ro_rrt-ud-train.conllu', 'validation': 'ro_rrt-ud-dev.conllu', 'test': 'ro_rrt-ud-test.conllu'}

In [ ]:
dataset_with_diac = load_dataset('./datasets/diac', data_files=data_files)

In [ ]:
dataset_with_diac['train'][0]

In [ ]:
# create a new dataset without diacritics
dataset_without_diac = {}

for dataset_split in ['train', 'validation', 'test']:
  dataset_without_diac[dataset_split] = strip_romanian_diacritics_from_dataset_sts(dataset_with_diac[dataset_split])

  # ner
  # with open(os.path.join('./datasets/nodiac', f'{dataset_split}_dataset.json'), 'w', encoding='utf-8') as f:
  #     json.dump(dataset_without_diac[dataset_split], f, ensure_ascii=False, indent=4)

  # sts
  df = pd.DataFrame(dataset_without_diac[dataset_split])
  df.to_csv(os.path.join('./datasets/nodiac', f'{dataset_split}_dataset.tsv'), sep="\t")


In [ ]:
# check it works
data_files = {'train': 'train_dataset.tsv', 'validation': 'validation_dataset.tsv', 'test': 'test_dataset.tsv'}
ds = load_dataset('./datasets/nodiac', data_files=data_files)
ds['train'][0]

In [ ]:
def strip_romanian_diacritics_from_dataset_pos(file):
    dataset_without_diac = []

    with open(file, "r", encoding="utf8") as f:
        text = f.readlines()
        dataset_without_diac = strip_romanian_diacritics_from_tokens(text)

    return dataset_without_diac


In [ ]:
for dataset_split in ['train', 'validation', 'test']:
  dataset_without_diac = strip_romanian_diacritics_from_dataset_pos("/content/datasets/diac/" + data_files[dataset_split])

  with open(os.path.join('./datasets/nodiac', f'{dataset_split}_dataset.conllu'), 'w', encoding='utf-8') as f:
    f.write(''.join(dataset_without_diac))
